In [9]:
!pip install -r "/kaggle/input/datasets/eces99/bachelor-project-ece/requirements.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers[torch] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.4 MB/s eta 0:00:0000:010:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.

In [10]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


True

In [11]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [16]:
import shutil
import os

# Copy everything to writable location
shutil.copytree(
    "/kaggle/input/datasets/eces99/bachelor-project-ece",
    "/kaggle/working/project",
    dirs_exist_ok=True
)

# Verify
print(os.listdir("/kaggle/working/project"))

['data', 'src', 'requirements.txt', 'training_models']


In [18]:
import os
os.chdir("/kaggle/working/project/training_models")

# Verify
print(os.getcwd())
print(os.path.exists("../data/spider/database"))

/kaggle/working/project/training_models
True


In [20]:
with open("/kaggle/working/project/training_models/train_all.py", "r") as f:
    content = f.read()

# Fix Model 2 loss
content = content.replace(
    "loss       = losses.MultipleNegativesRankingLoss(model)",
    "loss       = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)",
    1  # only replace first occurrence = Model 2
)

# Fix Model 3 loss
content = content.replace(
    "loss       = losses.MultipleNegativesRankingLoss(model)",
    "loss       = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)",
    1  # only replace second occurrence = Model 3
)

# Fix Model 2 examples - remove negatives, positives only
content = content.replace(
    '''def build_random_neg_examples(train_qs, schemas, neg_per_query):
    """
    (query, correct_schema) label=1.0
    (query, random_wrong_schema) × N label=0.0
    """
    all_dbs  = list(schemas.keys())
    examples = []

    for q in train_qs:
        correct_db = q["db_id"]
        if correct_db not in schemas:
            continue

        examples.append(InputExample(
            texts=[q["question"], schemas[correct_db]], label=1.0))

        wrong = [d for d in all_dbs if d != correct_db]
        for db in random.sample(wrong, min(neg_per_query, len(wrong))):
            examples.append(InputExample(
                texts=[q["question"], schemas[db]], label=0.0))

    return examples''',
    '''def build_random_neg_examples(train_qs, schemas):
    examples = []
    for q in train_qs:
        correct_db = q["db_id"]
        if correct_db not in schemas:
            continue
        examples.append(InputExample(texts=[q["question"], schemas[correct_db]]))
    return examples'''
)

# Fix Model 2 call to remove neg_per_query arg
content = content.replace(
    "examples   = build_random_neg_examples(train_qs, schemas, NEG_RANDOM)",
    "examples   = build_random_neg_examples(train_qs, schemas)"
)

# Fix Model 3 - triplets instead of pairs
content = content.replace(
    '''        # positive
        examples.append(InputExample(
            texts=[question, schemas[correct_db]], label=1.0))

        # fuse BM25 + TF-IDF scores, pick top wrong DBs as hard negatives
        bm25_scores  = _minmax_normalize(bm25.score(question))
        tfidf_scores = _minmax_normalize(tfidf.score(question))
        fused = {
            db: (bm25_scores.get(db, 0.0) + tfidf_scores.get(db, 0.0)) / 2
            for db in all_dbs if db != correct_db
        }
        hard_negs = sorted(fused.items(), key=lambda x: x[1], reverse=True)
        hard_negs = [db for db, _ in hard_negs[:neg_per_query]]

        for db in hard_negs:
            examples.append(InputExample(
                texts=[question, schemas[db]], label=0.0))''',
    '''        bm25_scores  = _minmax_normalize(bm25.score(question))
        tfidf_scores = _minmax_normalize(tfidf.score(question))
        fused = {
            db: (bm25_scores.get(db, 0.0) + tfidf_scores.get(db, 0.0)) / 2
            for db in all_dbs if db != correct_db
        }
        hard_negs = [db for db, _ in sorted(fused.items(), key=lambda x: x[1], reverse=True)[:neg_per_query]]
        for hard_neg_db in hard_negs:
            examples.append(InputExample(
                texts=[question, schemas[correct_db], schemas[hard_neg_db]]
            ))'''
)

# Fix hyperparameters
content = content.replace("EPOCHS_SBERT   = 1", "EPOCHS_SBERT   = 3")
content = content.replace("BATCH_SIZE     = 16", "BATCH_SIZE     = 64")
content = content.replace("NEG_HARD       = 3", "NEG_HARD       = 7")

# Fix MLP to use schemas_preprocessed
content = content.replace(
    "train_mlp(bm25, tfidf, semantic, raw_schemas, train_qs,",
    "train_mlp(bm25, tfidf, semantic, schemas_preprocessed, train_qs,"
)

with open("/kaggle/working/project/training_models/train_all.py", "w") as f:
    f.write(content)

print("Done — verifying key changes:")
# Quick check
for line in content.split("\n"):
    if any(x in line for x in ["EPOCHS_SBERT", "BATCH_SIZE", "NEG_HARD", "CachedMultiple", "schemas_preprocessed"]):
        print(" ", line.strip())

Done — verifying key changes:
  EPOCHS_SBERT   = 3
  BATCH_SIZE     = 64
  NEG_HARD       = 7     # hard negatives per query for hard-neg fine-tune
  dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
  loss       = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)
  print(f"  Epochs     : {EPOCHS_SBERT}")
  epochs=EPOCHS_SBERT,
  warmup_steps=_warmup_steps(dataloader, EPOCHS_SBERT),
  examples   = mine_hard_negatives(train_qs, schemas, bm25, tfidf, NEG_HARD)
  dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
  loss       = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)
  print(f"  Epochs     : {EPOCHS_SBERT}")
  epochs=EPOCHS_SBERT,
  warmup_steps=_warmup_steps(dataloader, EPOCHS_SBERT),
  schemas_preprocessed = load_schemas(DATABASE_DIR, preprocessor=p)
  bm25     = LexicalSelector(schemas_preprocessed, preprocessor=p, variant="okapi")
  tfidf    = TFIDFSelector(schemas_preprocessed, preprocessor=p, ng

In [25]:
!cd /kaggle/working/project && python training_models/train_all.py

2026-03-30 20:15:47.956398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774901747.984789     433 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774901747.993464     433 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774901748.013582     433 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774901748.013610     433 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774901748.013614     433 computation_placer.cc:177] computation placer alr

In [26]:
import shutil
shutil.make_archive("/kaggle/working/trained_models", "zip", "/kaggle/working/project/models")
print("Done — ready to download")

Done — ready to download


In [23]:
with open("/kaggle/working/project/training_models/train_all.py", "r") as f:
    content = f.read()

content = content.replace(
    "    # # Model 2: positives only + CachedMNRL\n    # train_finetuned(train_qs, dev_qs, raw_schemas, CKPT_FINETUNED)",
    "    # Model 2: positives only + CachedMNRL\n    train_finetuned(train_qs, dev_qs, raw_schemas, CKPT_FINETUNED)"
)

content = content.replace(
    "    # # Model 3: hard negative triplets + CachedMNRL\n    # train_hardneg(train_qs, dev_qs, raw_schemas, bm25, tfidf, CKPT_HARDNEG)",
    "    # Model 3: hard negative triplets + CachedMNRL\n    train_hardneg(train_qs, dev_qs, raw_schemas, bm25, tfidf, CKPT_HARDNEG)"
)

with open("/kaggle/working/project/training_models/train_all.py", "w") as f:
    f.write(content)

# Verify
with open("/kaggle/working/project/training_models/train_all.py", "r") as f:
    for line in f:
        if "train_finetuned" in line or "train_hardneg" in line:
            print(repr(line))

'def train_finetuned(train_qs, dev_qs, schemas, output_dir):\n'
'def train_hardneg(train_qs, dev_qs, schemas, bm25, tfidf, output_dir):\n'
'    train_finetuned(train_qs, dev_qs, raw_schemas, CKPT_FINETUNED)\n'
'    train_hardneg(train_qs, dev_qs, raw_schemas, bm25, tfidf, CKPT_HARDNEG)\n'


In [24]:
with open("/kaggle/working/project/training_models/train_all.py", "r") as f:
    content = f.read()
print(content)

import os
import sys
import random
from pathlib import Path

sys.path.append(os.path.join(os.path.dirname(__file__), '..'))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation

from src.selector.schema_repr import load_schemas, load_queries, Preprocessor
from src.selector.lexical     import LexicalSelector
from src.selector.statistical import TFIDFSelector
from src.selector.semantical  import SemanticSelector
from mlp_fusion               import FusionMLP, FusionDataset, train_mlp
from hybrid                   import _minmax_normalize

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


TRAIN_PATH   = "data/spider/train_spider.json"
DEV_PATH     = "data/spider/dev.json"
DATABASE_DIR = "data/spider/database"

CKPT_FINETUNED = "models/gte-small-finetuned"
CKPT_HARDNEG   = "models/gte-small-hardneg"
CKPT_MLP       = "models/mlp_fusion.pt"

BASE_MODEL     = "t